# 🧪 Aula 11 — Modelagem Híbrida e Caixa Cinza

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `cstr_hibrido.csv` — reator CSTR em estado estacionário (10.000 pontos)

---

## Contexto

Você recebeu dados de um **reator CSTR** operando em estado estacionário. O balanço de massa é:

$$C_A = \frac{F \cdot C_{A0}}{F + V \cdot k}$$

Mas a constante cinética $k$ é **desconhecida** e varia com a temperatura.

## 3.1 — Modelo Híbrido: 5 Passos

Combine o **balanço de massa** (física) com um **modelo ML** (dados) para estimar $k$.

### Passo 1: Escrever a equação para calcular k dos dados

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

V = 100.0  # L (volume do reator)

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula11/cstr_hibrido.csv"
df = pd.read_csv(URL)
print(df.head())

# Do balanço: C_A = F*C_A0/(F+V*k)  →  k = F*(C_A0 - C_A)/(V*C_A)
df['k_real'] = df['F_alimentacao_L_min'] * (df['C_A0_mol_L'] - df['C_A_mol_L']) / (V * df['C_A_mol_L'])
print(f"k_real: média={df['k_real'].mean():.4f}  std={df['k_real'].std():.4f}")

### Passo 2: Visualizar k vs T (deve seguir Arrhenius)

In [ ]:
plt.figure(figsize=(10, 4))
plt.scatter(df['T_reator_C'], df['k_real'], alpha=0.3, s=5)
plt.xlabel('Temperatura (°C)')
plt.ylabel('k_real (min⁻¹)')
plt.title('Constante cinética vs Temperatura — crescimento exponencial (Arrhenius)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Passo 3: Treinar ML para estimar k

In [ ]:
features = ['T_reator_C', 'F_alimentacao_L_min', 'C_A0_mol_L']
X = df[features]
y_k = df['k_real']
X_train, X_test, yk_train, yk_test = train_test_split(X, y_k, test_size=0.2, random_state=42)

model_k = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
model_k.fit(X_train, yk_train)
print(f"R² do modelo de k (treino): {r2_score(yk_train, model_k.predict(X_train)):.4f}")

### Passo 4: Predizer C_A híbrido

In [ ]:
k_pred = model_k.predict(X_test)
C_A_hibrido = (X_test['F_alimentacao_L_min'] * X_test['C_A0_mol_L']) / (V * k_pred + X_test['F_alimentacao_L_min'])

C_A_real = df.loc[X_test.index, 'C_A_mol_L']
rmse_hib = np.sqrt(mean_squared_error(C_A_real, C_A_hibrido))
r2_hib = r2_score(C_A_real, C_A_hibrido)
print(f"Híbrido: RMSE={rmse_hib:.4f}  R²={r2_hib:.4f}")

### Passo 5: Comparar com caixa branca e preta

In [ ]:
# Caixa preta: ML direto para C_A
model_d = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
model_d.fit(X_train, df.loc[X_train.index, 'C_A_mol_L'])
C_A_direto = model_d.predict(X_test)

# Caixa branca: k constante = média
k_med = df['k_real'].mean()
C_A_branca = (X_test['F_alimentacao_L_min'] * X_test['C_A0_mol_L']) / (V * k_med + X_test['F_alimentacao_L_min'])

def eval_model(nome, y_real, y_pred):
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    print(f"{nome:24s} RMSE={rmse:.4f}  R²={r2:.4f}")

print(f"{'Modelo':24s} {'RMSE':>7s} {'R²':>7s}")
print('-' * 42)
eval_model('Caixa Branca (k const)', C_A_real, C_A_branca)
eval_model('Caixa Preta (ML direto)', C_A_real, C_A_direto)
eval_model('Híbrido (balanço+ML)', C_A_real, C_A_hibrido)

### ✏️ Pausa reflexiva (2 min)

O $k$ calculado dos dados tem ruído — **por que o ML ajuda a suavizar?**

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Extrapolação

O dataset `cstr_hibrido_extrapolacao.csv` tem temperaturas de **125-135 °C** — fora da faixa de treino (70-120 °C).

In [ ]:
URL_e = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula11/cstr_hibrido_extrapolacao.csv"
df_e = pd.read_csv(URL_e)
df_e['k_real'] = df_e['F_alimentacao_L_min'] * (df_e['C_A0_mol_L'] - df_e['C_A_mol_L']) / (V * df_e['C_A_mol_L'])

X_e = df_e[features]

# Predizer com os 3 modelos (já treinados acima)
C_A_real_e = df_e['C_A_mol_L']

# Híbrido
k_e = model_k.predict(X_e)
C_hib_e = (X_e['F_alimentacao_L_min']*X_e['C_A0_mol_L']) / (V*k_e + X_e['F_alimentacao_L_min'])

# Caixa preta (ML direto)
C_dir_e = model_d.predict(X_e)

# Caixa branca (k const)
C_bran_e = (X_e['F_alimentacao_L_min']*X_e['C_A0_mol_L']) / (V*k_med + X_e['F_alimentacao_L_min'])

for nome, yp in [('Caixa Branca', C_bran_e), ('Caixa Preta', C_dir_e), ('Híbrido', C_hib_e)]:
    rmse = np.sqrt(mean_squared_error(C_A_real_e, yp))
    r2 = r2_score(C_A_real_e, yp)
    print(f"{nome:16s} extrapolação: RMSE={rmse:.4f}  R²={r2:.4f}")

# Qual modelo degradou mais? O híbrido ainda é razoável? Por quê?

### 🧠 Desafio extra (NT)

O balanço de massa usado no híbrido assume volume constante $V$. Se $V$ variar com a vazão (reator não-ideal), o modelo híbrido degrada? Como melhorar?

> _Escreva aqui..._

---

## Checklist do Modelo Híbrido

- [ ] Equações do balanço documentadas
- [ ] Parâmetro desconhecido identificado (k)
- [ ] k calculado dos dados
- [ ] ML treinado para estimar k
- [ ] Predição híbrida vs real (RMSE/R²)
- [ ] Comparação com caixa branca e preta
- [ ] Extrapolação testada
- [ ] Conclusão escrita